In [1]:
%load_ext autoreload
%autoreload 2

import time
import random
from copy import deepcopy
from itertools import tee
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch as th
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from toolz.curried import *
from toolz.sandbox.core import unzip

from mmidas.nn_model import make_mmidas
from mmidas.model import Net
from mmidas.train import generic_train
from mmidas._data_util import make_loaders, viz, make_mnist
from mmidas._utils import randomize, shuffle, npercent

In [2]:
train_loader, val_loader, test_loader = make_loaders('mnist', batch_size=32)
train_mnist, val_mnist, test_mnist = make_mnist(datasets.MNIST) # batch with partition()

In [3]:
net = Net()
opt = th.optim.SGD(net.parameters(), lr=1e-3)
metrics = generic_train(net, opt, train_loader, val_loader, epochs=10)

TypeError: generic_train() missing 3 required positional arguments: 'loss_fn', 'device', and 'f'

In [4]:
device = 'cpu'
n_categories = 10
state_dim = 2
input_dim = 784
n_arm = 2
lr = 1e-3

model = make_mmidas(10, 2, 784, n_arm=n_arm, device=device).to(device)
opt = th.optim.Adam(model.parameters(), lr=1e-3)

In [37]:
def simplify_mmidas(f):
    def g(x):
        return f([x for _ in range(f.n_arm)], temp=1)
    return g


def mmidas_loss(model):
    def l(y_hat, y):
        x_recs, _, _, _, cs, _, c_smps, s_means, s_logvars, _ = y_hat
        return model.loss(x_recs, [], [], y, s_means, s_logvars, cs, c_smps, 0.0)
    return l



x, y = first(train_loader)

simplify_mmidas(model)(x)

([tensor([[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]], grad_fn=<ReluBackward0>),
  tensor([[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]], grad_fn=<ReluBackward0>)],
 [],
 [],
 [tensor([[-6.9438e-01, -4.4627e-01, -4.8869e-01, -7.6370e-01,  3.0297e-01,
           -6.2306e-01,  2.1749e-01,  1.3373e+00, -2.7209e-01,  1.2301e-01],
          [ 4.8754e-01, -4.4627e-01, -4.8869e-01, -5.1141e-01, -6.0909e-01,
           -6.2306e-01, -6.1539e-01, -6.0252e-01, -4.8615e-02,  1.4672e-01],
          [ 3.8944e-01,  9.4229e-01, -4.8869e-01,  1.2062e+00,  1.7517e+00,
           -6.2306e

In [ ]:
metrics = generic_train(model, opt, train_loader, None, 10, mmidas_loss(model), device, simplify_mmidas)

Epoch 1/10 | loss: 3826524.0000 | acc: 0.0000:  61%|██████    | 908/1500 [00:07<00:04, 126.59it/s]    

In [ ]:
tic = time.time()
loss_naive = model.loss_naive(cs)
t1 = time.time() - tic

tic = time.time()
loss_vec = model.loss_vectorize(cs)
t2 = time.time() - tic

print(f"Naive loss: {loss_naive.item()}")
print(f"Vectorized loss: {loss_vec.item()}")
print(f"Relative error: {th.norm(loss_naive - loss_vec) / th.norm(loss_naive)}\n")


print(f"Naive loss computation took: {t1}s")
print(f"Vectorized loss computation took: {t2}s")
print(f"Speedup: {100 * (t1 - t2) / t1:.2f}%")